# Behavioral: HorizonScale — Prophet at Scale, 6k Endpoints

| Story | Core Signal |
|-------|-------------|
| Prophet at Scale | ML engineering + distributed forecasting |
| 6k Endpoint Telemetry | System design + observability at scale |
| Citi ML Forecasting | Cross-functional stakeholder + model ops |
| Design Decisions | Trade-off reasoning and outcome ownership |
| Follow-up Defense | Technical depth under pressure |

**Purpose**: Structure real project experience into crisp STAR narratives that communicate technical ownership, scale, and measurable outcomes. Each story answers: *What was hard? What did you do? How do you know it worked?*

```
STAR format:
  Situation: context, scale, constraints
  Task:      your specific responsibility
  Action:    what you built/changed/decided — concrete and technical
  Result:    quantified outcome + business impact
```

## Visual Model — Story Structure

```
STORY ANATOMY
──────────────
  Opening hook (1 sentence): scale + what you owned
  │
  ├─ Situation: team size, system scale, business context
  ├─ Problem:   what was failing/missing/slow
  ├─ Your role: individual contribution (not "we")
  ├─ Actions:   3-4 concrete technical decisions with tradeoffs
  └─ Results:   numbers — latency, accuracy, cost, revenue, time

SIGNAL MAP (what interviewers listen for)
──────────────────────────────────────────
  Scale awareness:   "at 6k endpoints, polling failed because..."
  Trade-off thinking: "we chose X over Y because..."
  Ownership:         "I designed", "I pushed back", "I proposed"
  Outcome clarity:   "reduced MAPE from 18% to 11%"
  Failure awareness: "first attempt failed because..."

COMMON INTERVIEW PROMPTS → STORY MAPPING
──────────────────────────────────────────
  "Tell me about a complex technical project"  → Prophet at scale
  "Describe a system you designed from scratch" → Telemetry / ML pipeline
  "Cross-functional challenge"                 → Citi stakeholder story
  "When you had to make a difficult trade-off" → Architecture decisions
  "Most impactful thing you shipped"           → ML forecasting outcomes
```

## Setup — Story Delivery Framework

In [ ]:
# Story delivery framework: print full STAR story with timing targets

def print_story(title, situation, task, actions, result, timing_s=120):
    """Format and print a STAR story with structure markers."""
    print(f"\n{'='*60}")
    print(f"STORY: {title}")
    print(f"Target: ~{timing_s}s verbal delivery")
    print(f"{'='*60}")

    print("\n[SITUATION] ~20s")
    print(situation)

    print("\n[TASK] ~10s")
    print(task)

    print("\n[ACTIONS] ~60s")
    for i, action in enumerate(actions, 1):
        print(f"  {i}. {action}")

    print("\n[RESULT] ~30s")
    print(result)

def print_followup(question, answer):
    print(f"\nQ: {question}")
    print(f"A: {answer}")

print("Story framework loaded.")

## Decision Map — Story Selection

```
Interview question type → which story to use
│
├─ "Technical complexity / scale"
│   └─ → Prophet at Scale: 6k endpoints, parallel training, MAPE improvement
│
├─ "System design / architecture"
│   └─ → Telemetry infrastructure: push→pull, CloudWatch ETL, cost reduction
│
├─ "Cross-functional / stakeholder"
│   └─ → Citi ML forecasting: worked with trading desk, model transparency
│
├─ "Conflict / disagreement / pushback"
│   └─ → Architecture trade-off: pushed back on naive approach
│
├─ "Failure / learned from mistake"
│   └─ → First Prophet attempt: batch job too slow, pivoted to parallelism
│
└─ "Impact / most proud of"
    └─ → ML pipeline accuracy improvement + production A/B test results

Story length guide:
  1-2 minutes:  screening / phone screen
  2-3 minutes:  onsite behavioral panel
  Stop at result, invite follow-up: "Happy to go deeper on any part"
```

## Pattern 1 — Prophet at Scale: Core Story

In [ ]:
print_story(
    title="Prophet Forecasting at Scale — HorizonScale",

    situation="""
At HorizonScale, I was the lead data engineer on a forecasting platform
that powered inventory and capacity planning for ~6,000 retail locations.
The business needed weekly demand forecasts per location-SKU, roughly
1.2 million models in total. The existing system ran a single-threaded
Python loop using Facebook Prophet — it took 36 hours to complete one
full training cycle, which meant forecasts were perpetually stale.
""",

    task="""
I owned the end-to-end redesign: from pipeline architecture to
model training strategy to production deployment. The goal was
to get weekly forecasts done in under 4 hours with no accuracy regression.
""",

    actions=[
        """Profiled the bottleneck: Prophet training is CPU-bound (Stan MCMC),
   not I/O. Moving to distributed Spark with 1 model per task achieved
   ~40x speedup. I rejected GPU acceleration early — Prophet doesn't
   parallelize within a model, so vertical scaling was a dead end.""",

        """Designed a hierarchical feature store: global trend features at
   chain level, regional seasonality at region level, and location-specific
   regressors (promotions, local events) at the leaf. This let Prophet
   warm-start from shared priors, cutting per-model training time by 30%.""",

        """Implemented a model selection layer: for SKUs with <26 weeks of
   history, Prophet was replaced with a simple seasonal decomposition
   (STL + ETS). This prevented overfitting on sparse series and reduced
   MAPE on new-store SKUs from 28% to 14%.""",

        """Built a backtesting harness with walk-forward validation (52-week
   train, 4-week hold-out, 48 folds) that ran on every model retrain.
   This caught a silent 3% MAPE degradation after a hyperparameter
   change that no one had noticed in production.""",
    ],

    result="""
Full training cycle: 36 hours → 3.2 hours (11x improvement).
MAPE (test set): 18.4% → 11.1% across all SKUs.
New-store MAPE: 28% → 14% (model selection routing).
Pipeline cost: ~$800/week (EMR) vs $0 (was in-office server farm,
but with zero reliability SLA). First time the business got
Monday-morning forecasts ready before market open.
The backtesting harness caught 2 silent regressions in 6 months.
""",
    timing_s=120
)

## Pattern 2 — Follow-up Defense: Prophet Architecture

In [ ]:
# Common follow-up questions and tight, technical answers

followups = [
    (
        "Why Prophet and not a simpler model? Or LightGBM?",
        """Prophet's interpretability was a hard business requirement — the retail
team needed to explain 'why did we forecast 500 units for Black Friday'
to store managers. LightGBM would have been a black box there.
We did use LightGBM for a secondary ranking model (which locations to
prioritize restocking), but the primary demand signal stayed with Prophet."""
    ),
    (
        "How did you handle model failures / timeouts in Spark?",
        """Each Spark task had a 10-minute timeout. On failure, the task logged
the model key and fell back to the prior week's forecast. After the run,
a reconciliation job identified failed models and either retried with
relaxed priors or escalated to manual review if failure rate exceeded 0.5%."""
    ),
    (
        "What's MAPE and why did you choose it?",
        """MAPE = mean absolute percentage error. It's scale-independent,
which matters when you're comparing a 10-unit SKU to a 10,000-unit SKU.
We also tracked sMAPE (symmetric MAPE) for SKUs where actual=0 occasionally,
since MAPE blows up on zero actuals. For planning, we ultimately reported
bias (over/under forecast) because the business cared more about
consistent directional error than absolute magnitude."""
    ),
    (
        "What would you do differently?",
        """I'd implement a model registry earlier — we tracked model versions
in a spreadsheet for the first 3 months, which made debugging regressions
painful. I'd also add feature importance logging from day one, since
we had several conversations with stakeholders about 'what's driving this
forecast' that required one-off analyses we hadn't pre-built."""
    ),
    (
        "How did you validate that parallelization didn't change model outputs?",
        """We ran a shadow period: the parallel pipeline ran alongside the old
sequential one for 2 weeks, and we diffed outputs row-by-row.
Differences were within floating-point tolerance (< 1e-6) since
Prophet is deterministic given the same seed and Stan version.
The only non-trivial delta came from a Stan version bump — we pinned
it and re-validated."""
    ),
]

for q, a in followups:
    print_followup(q, a)

## Pattern 3 — Citi ML Forecasting Story

In [ ]:
print_story(
    title="Citi ML Forecasting — Cross-Functional Stakeholder Story",

    situation="""
At Citi, I supported the fixed income trading desk with a cash flow
forecasting model. The trading desk had been using a rules-based
Excel model for 8 years. A new head of desk wanted to bring in ML
to improve forecast accuracy, but the quants were skeptical —
they needed to trust and explain the model to regulators.
""",

    task="""
I was the technical lead on a 3-person data science team. My
specific role was to design the feature pipeline and model
evaluation framework, and to present technical findings to both
the trading desk and a compliance review panel.
""",

    actions=[
        """Ran a feature audit first: I catalogued 47 candidate features the
   quants proposed and ran correlation + Granger causality tests to
   identify the 12 features with genuine predictive signal. This
   immediately built credibility — I could say 'your intuition about
   overnight repo rates is confirmed; the LIBOR spread is noise.'""",

        """Chose a gradient boosted tree (XGBoost) with SHAP explanations
   over a neural network, specifically because SHAP let us show
   each trader which features drove each day's forecast. The compliance
   team required explainability — this was a non-negotiable constraint.""",

        """Built a temporal walk-forward validation with strict lookahead
   prevention: trained on T-252 to T-63 trading days, validated on
   T-63 to T-21, tested on T-21 to T. This prevented the common
   mistake of random-splitting financial time series (data leakage).""",

        """Created a model monitoring dashboard in Tableau showing daily
   forecast vs actual, feature drift metrics (PSI for each input),
   and an alert when MAPE exceeded the 3-month rolling average by 2σ.
   This gave the desk confidence that we'd catch model degradation
   before it caused a significant mis-hedge.""",
    ],

    result="""
Model MAPE: 12.3% vs 19.7% for the rules-based Excel model
(37% improvement). Compliance approved after the first panel review —
the SHAP explainability output was key. The desk adopted it for
daily hedging decisions within 6 weeks of go-live. One senior quant
who was initially skeptical became the internal champion after the
SHAP analysis confirmed his hypothesis about repo rate sensitivity.
""",
    timing_s=150
)

## Pattern 4 — Architecture Trade-off Story

In [ ]:
print_story(
    title="Architecture Trade-off: Pushback on Naive Solution",

    situation="""
During the HorizonScale ML platform redesign, the product manager
pushed for a fully automated "one-click retrain" feature where
any engineer could trigger a full model retrain from a UI button.
The existing compute budget was tight — a full retrain cost ~$2,400
in EMR costs.
""",

    task="""
I was responsible for the training pipeline, and I needed to either
implement the feature as requested or make the case for an alternative
that balanced the PM's goal with operational safety.
""",

    actions=[
        """I ran a cost-of-error analysis: an accidental full retrain
   during peak hours would create a 3-hour window with stale forecasts
   and cost $2,400 in compute. I framed this as a risk the PM should
   explicitly sign off on, not a technical 'no.'""",

        """I proposed a tiered retrain system:
   - Tier 1 (free, instant): re-score existing models with fresh features
   - Tier 2 ($200, 30 min): retrain a specified subset (e.g., one region)
   - Tier 3 ($2.4k, 3 hrs): full retrain, requires sign-off + scheduled window.
   The PM wanted Tier 1 and 2 — I built those. Tier 3 stayed scheduled.""",

        """To address the underlying need (faster response to data drift),
   I implemented continuous drift detection using PSI on input features.
   When drift exceeded a threshold, it automatically triggered a Tier 2
   retrain for the affected regions without requiring manual intervention.""",
    ],

    result="""
The PM got the responsiveness they wanted (average drift response
time went from 2 weeks to 48 hours), we eliminated accidental
full retrains, and compute costs dropped 15% because targeted
Tier 2 retrains replaced full cycles for regional drift events.
The PM later said it was a better solution than what they'd originally asked for.
""",
    timing_s=90
)

## Pattern 5 — Rapid Story Variants (Short Form)

In [ ]:
# Short-form stories for screening or "give me a quick example" questions

short_stories = {
    "Failure / learn": """
Early in the Prophet project, I optimized our cluster for 400 large Spot instances
to maximize parallelism. What I hadn't accounted for: Spot reclamation rate was 8%
in the instance family I'd chosen, and Spark task retries on those failures added
20 minutes back to the runtime. I switched to a mix of 80% on-demand (smaller) +
20% Spot, which added $300/run but cut failure-induced overhead from 20 minutes
to under 3 minutes. Lesson: reliability cost is real and belongs in the cost model.
""",

    "Speed / urgency": """
A production model at HorizonScale started returning nonsensical forecasts at
2am on a Monday before a major retail promotion. I got paged, identified that
a upstream ETL had started filling missing values with 0 instead of NULL
(a schema change no one had documented). I rolled back to last week's model
within 40 minutes, drafted a post-mortem, and the next sprint we added schema
drift detection to the feature pipeline. Forecasts were live before market open.
""",

    "Influence without authority": """
The analytics team at Citi stored model training data in a shared Snowflake
schema without version control or access logging. I had no authority over
their data practices, but I needed reproducible training sets.
I built a lightweight data snapshot tool that wrote training set manifests
to S3 before each run, shared it with the analytics team, and showed how
it helped them too (they'd had a model regression they couldn't reproduce).
Within 2 months it was standard practice for both teams.
""",

    "Working with ambiguity": """
When I joined HorizonScale, the forecasting requirement was 'make it more accurate.'
Before writing any code, I spent a week understanding the downstream consumer:
supply chain planners who used forecasts to set reorder points. I learned they
cared more about reducing stockouts than average MAPE — so the right metric
was service level (% of weeks with no stockout), not MAPE.
Reframing the objective changed model design: we optimized for P90 forecast
(overestimate safely) rather than mean, which is what the planners actually needed.
""",
}

for label, story in short_stories.items():
    print(f"\n{'─'*50}")
    print(f"SHORT STORY: {label}")
    print(f"{'─'*50}")
    print(story.strip())

## Full Decision Map

```
STORY SELECTION + DELIVERY GUIDE
──────────────────────────────────
Prompt → Story
  Technical complexity    → Prophet at scale (parallelism, hierarchy, backtesting)
  Cross-functional        → Citi: SHAP for compliance, quant skeptic won over
  Trade-off / pushback    → Tiered retrain: cost-risk analysis, better alternative
  Failure / learning      → Spot reclamation lesson: cost model must include reliability
  Ambiguity / scoping     → MAPE vs service level: understand the downstream consumer
  Speed / incident        → 2am forecast incident: rollback in 40 min, post-mortem
  Influence w/o authority → Data snapshot tool: built value, shared, adopted organically

DELIVERY CHECKLIST
  ✓ Open with scale: "6,000 locations", "1.2 million models", "2am page"
  ✓ Be specific about your role: "I designed", "I pushed back", "I built"
  ✓ Name the trade-off: "X over Y because Z constraint"
  ✓ Lead with the number in the result: "reduced from 36h to 3.2h"
  ✓ Invite follow-up: "happy to go deeper on any of those decisions"

ANTI-PATTERNS
  ✗ "We" throughout (use I for your contributions, we for team)
  ✗ Vague result: "improved significantly" → say the number
  ✗ No trade-off: everything just worked perfectly (not credible)
  ✗ Too much context, not enough action: flip the ratio 30/70
  ✗ Over-indexing on technology names without explaining why
```

## Cheat Sheet

```
STORY QUICK REFERENCE
──────────────────────
Prophet at Scale (HorizonScale):
  S: 6k endpoints, 1.2M models, 36h training cycle
  T: Lead engineer — redesign pipeline end-to-end
  A: Spark parallel (40x), hierarchical feature store, model selection (Prophet vs STL+ETS), backtesting harness
  R: 36h → 3.2h, MAPE 18% → 11%, new-store 28% → 14%

Citi ML Forecasting:
  S: Fixed income trading desk, rules-based Excel model, compliance constraints
  T: Tech lead — feature pipeline + evaluation + compliance presentation
  A: Feature audit (47→12), XGBoost + SHAP (explainability), temporal validation, PSI monitoring dashboard
  R: MAPE 19.7% → 12.3%, compliance approved first review, adopted in 6 weeks

Architecture Trade-off:
  S: PM wanted one-click full retrain, $2.4k/run, reliability risk
  T: Pipeline owner — implement or propose alternative
  A: Cost-of-error framing, tiered retrain system (free/partial/full), PSI drift auto-trigger
  R: Drift response 2w → 48h, compute -15%, no accidental full retrains

KEY NUMBERS
  6,000 retail endpoints
  1.2 million models
  36h → 3.2h (11x) training
  MAPE 18.4% → 11.1%
  $2,400 per full retrain
  ~37% MAPE improvement at Citi
  40 minutes to rollback in incident
```

## Summary Map

```
BEHAVIORAL STORIES — ONE-PAGE SUMMARY
───────────────────────────────────────

CORE STORIES
  1. Prophet at scale:   parallelism, hierarchy, model selection, backtesting
  2. Citi forecasting:   feature audit, SHAP explainability, compliance navigation
  3. Architecture trade-off: tiered retrain, cost-risk framing, drift detection

SHORT STORIES (60-90s)
  Failure:     Spot reclamation lesson → reliability cost in cost model
  Incident:    2am forecast bug → rollback 40min, schema drift detection added
  Influence:   data snapshot tool → shared value, organically adopted
  Ambiguity:   MAPE vs service level → understand downstream consumer first

DELIVERY FORMAT
  Scale → Problem → My role → Actions (3-4) → Numbers
  2 minutes for onsite; 90s for screens; invite follow-up at end

SIGNALS TO DEMONSTRATE
  ✓ Technical depth (can go deeper on any decision)
  ✓ Scale awareness (numbers and constraints)
  ✓ Trade-off ownership (chose X over Y because)
  ✓ Business orientation (understood downstream impact)
  ✓ Learning from failure (didn't hide it)
```